In [ ]:
import polars as pl
import polars.selectors as cs
import time
import os

# Python Polars Essential Training - Day 1

Welcome! Today you'll learn the foundation of using Polars. We'll go over all the base data structures and core concepts of Polars so you understand how to build your own queries.

Let's start with importing Polars and printing the versions to check if everything works as it should.

In [ ]:
pl.show_versions()

## Base Data Structures - Exercises

In the cell below, create 3 Series objects. Create one containing some strings you make up, some ints, and some floats.

In [ ]:
# Your code here

Now create a DataFrame out of these Series.

In [ ]:
# Your code here

Create a DataFrame straight from a Python dictionary.

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Creating Series</summary>

```python
series_names = pl.Series("name", ["Alice", "Bob", "Charlie"])
series_ages = pl.Series("age", [25, 30, 35])
series_scores = pl.Series("score", [85.5, 90.0, 78.5])
```
</details>

<details>
<summary>Show Answer: DataFrame from Series</summary>

```python
df_from_series = pl.DataFrame([series_names, series_ages, series_scores])
df_from_series
```
</details>

<details>
<summary>Show Answer: DataFrame from Dictionary</summary>

```python
df_from_dict = pl.DataFrame(
    {
        "name": ["David", "Eve", "Frank"],
        "age": [40, 45, 50],
        "score": [92.0, 88.5, 76.0],
    }
)
df_from_dict
```
</details>


## Input / Output

Writing all your data yourself takes a while and presumably you want to work on data that already exists. This is where the suite of IO-related methods comes in.

In the cells below, read and display `data/scientists.xlsx`.

In [ ]:
# Your code here

Load and display `data/animals.csv`.

**Hint**: This file uses `;` as separator, has a junk row after the header, and uses `"n/a"`, `"-"`, and `"NA"` as null values.

In [ ]:
# Your code here

And now finally we will transform one of the parquet files into a CSV. This can be done using `pl.read_parquet().write_csv()` or `pl.scan_parquet().sink_csv()`. Time the difference.

The first option reads all the data into memory and then writes it to disk. The second option uses the streaming engine and, while reading the data, already writes chunks of data to disk.

In [ ]:
# Your code here

### Time for the difference: Formats matter!

Let's see what the difference is in reading a csv file vs a parquet file.
Start with reading `data/yellow_tripdata_2025-01.parquet` and writing it to a csv file named `data/yellow_tripdata_2025-01.csv`. This makes sure it's the exact same data, and only the format is different.

After that, time the difference between reading `yellow_tripdata_2025-01.parquet` and reading `yellow_tripdata_2025-01.csv`.

Also compare the filesize with `os.path.getsize("filepath")`.

In [ ]:
# Your code here

In order to inspect the effect of the filetype on the data you've read, you can call `.schema` or `.dtypes` on the DataFrame. What's the difference between the schemas from a DataFrame that comes from parquet compared to a DataFrame that comes from csv?

In [ ]:
taxi_parquet.schema, taxi_csv.schema

### Answers

<details>
<summary>Show Answer: Reading Excel Files</summary>

```python
scientists = pl.read_excel("data/scientists.xlsx")
scientists
```
</details>

<details>
<summary>Show Answer: Reading CSV with Options</summary>

```python
animals = pl.read_csv(
    "data/animals.csv",
    separator=";",
    skip_rows_after_header=1,
    null_values=["n/a", "-", "NA"],
)
animals
```
</details>

<details>
<summary>Show Answer: Parquet to CSV Conversion</summary>

```python
import time

def benchmark(func, runs=3):
    total = 0
    for _ in range(runs):
        start = time.perf_counter()
        func()
        total += time.perf_counter() - start
    return total / runs

rw_avg = benchmark(lambda: pl.read_parquet("data/yellow_tripdata_2025-01.parquet").write_csv("data/yellow_tripdata_2025-01.csv"))
print(f"Read/Write time (avg): {rw_avg:.5f}s")

ss_avg = benchmark(lambda: pl.scan_parquet("data/yellow_tripdata_2025-01.parquet").sink_csv("data/yellow_tripdata_2025-01.csv"))
print(f"Scan/Sink time (avg): {ss_avg:.5f}s")
```

</details>

<details>
<summary>Show Answer: Format Comparison Benchmark</summary>

```python
start = time.perf_counter()
taxi_parquet = pl.read_parquet("data/yellow_tripdata_2025-01.parquet")
parquet_time = time.perf_counter() - start

start = time.perf_counter()
taxi_csv = pl.read_csv("data/yellow_tripdata_2025-01.csv")
csv_time = time.perf_counter() - start

print(f"Taxi dataset columns: {list(taxi_parquet.columns)}")
print(f"Reading parquet took {parquet_time:.3f}s and reading csv took {csv_time:.3f}s",
    f"Parquet file size: {os.path.getsize('data/yellow_tripdata_2025-01.parquet') / 1024 / 1024:.2f}MB, csv file size: {os.path.getsize('data/yellow_tripdata_2025-01.csv') / 1024 / 1024:.2f}MB",
)
```
</details>

<details>
<summary>Show Answer: Schema Inspection</summary>

```python
taxi_parquet.schema, taxi_csv.schema
```
</details>


### Cloud Storage

Real-world data often lives in the cloud (S3, Azure Blob, GCS).
Polars supports reading directly from cloud storage.

```python
# Example (requires authentication)
pl.read_parquet(
    "s3://my-bucket/data.parquet",
    storage_options={
        "aws_access_key_id": "...",
        "aws_secret_access_key": "...",
        "aws_region": "us-east-1",
    }
)
```

This works for both `read_parquet` (eager) and `scan_parquet` (lazy).


## Lazy vs Eager execution

Polars has two execution modes: lazy and eager.
Eager execution is when the code is executed immediately.
Lazy execution is when the code is not executed until you explicitly ask for it.
This is done by using `collect()`.

Let's see the difference in timing.
We'll do the same operation as before, but one eager and one lazy.

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Lazy vs Eager Execution</summary>

```python
def benchmark(func, runs=5):
    total = 0
    for _ in range(runs):
        start = time.perf_counter()
        func()
        total += time.perf_counter() - start
    return total / runs

eager_time = benchmark(
    lambda: pl.read_parquet("data/yellow_tripdata_2025-*.parquet").filter(pl.col("passenger_count") > 2)
)

lazy_time = benchmark(
    lambda: pl.scan_parquet("data/yellow_tripdata_2025-*.parquet").filter(pl.col("passenger_count") > 2).collect()
)

print(f"Eager took {eager_time:.5f}s and lazy took {lazy_time:.5f}s. That's a {eager_time / lazy_time:.2f}x speed-up!")

(
    pl.scan_parquet("data/yellow_tripdata_2025-*.parquet")
    .filter(pl.col("passenger_count") > 2)
    .show_graph()
)
```
</details>


## Contexts

Now that we've got some data to work with, it's time to start manipulating it.
The main contexts in Polars are `select`, `with_columns`, `filter`, `group_by` with `agg`, and `join`.
Let's explore them one by one.

### Select

Select is used to select columns from a DataFrame.
Select the `Name` and `Born` columns from the `scientists` DataFrame.

In [ ]:
# Your code here

### With Columns

`with_columns` is used to add or modify columns.
Add a column `age` to the `scientists` DataFrame. You can calculate it by taking the `Died` column and subtracting the `Born` column.

In [ ]:
# Your code here

### Filter

`filter` is used to select rows based on a condition.
Filter the `scientists` DataFrame to only include scientists born after 1900.

In [ ]:
# Your code here

### Group By and Agg

`group_by` is used to group rows that have the same values in specified columns into a single row.
`agg` is used to aggregate the results of the group by.

Let's use the `animals.csv` dataset for this.
Group by `Conservation_Status` and count the number of animals in each status.

In [ ]:
# Your code here

### Join

Real-world data often comes from multiple sources. We have a file `data/awards.csv` that contains the number of awards won by various scientists.

Your task:
1. Load `data/awards.csv`.
2. Perform a **left join** to combine it with your `scientists` DataFrame. We want to keep all scientists, even if they don't have an entry in the awards file.
3. Clean up: The join will result in `null` values for scientists not found in the awards file. Fill these `null`s with `0`.

In [ ]:
# Your code here

### Selectors

Typing out every column name is for people with too much time. Use `polars.selectors` (imported as `cs`) to select columns by type, name patterns, or other properties.

Tasks using the `animals` DataFrame:
1. Select all numeric columns.
2. Select the `Animal` column 
3. Select all columns that contain the word "Lifespan".

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Select</summary>

```python
scientists.select(["Name", "Born"])
```
</details>

<details>
<summary>Show Answer: With Columns</summary>

```python
scientists.with_columns((pl.col("Died") - pl.col("Born")).alias("age"))
```
</details>

<details>
<summary>Show Answer: Filter</summary>

```python
scientists.filter(pl.col("Born") > 1900)
```
</details>

<details>
<summary>Show Answer: Group By and Agg</summary>

```python
animals.group_by("Conservation_Status").agg(pl.len().alias("count"))
```
</details>

<details>
<summary>Show Answer: Join</summary>

```python
awards = pl.read_csv("data/awards.csv")
scientists.join(awards, on="Name", how="left").with_columns(
    pl.col("Major_Awards").fill_null(0)
)
```
</details>

<details>
<summary>Show Answer: Selectors</summary>

```python
# 1. Select all numeric columns
print(animals.select(cs.numeric()))

# 2. Select the Animal column
print(animals.select("Animal"))

# 3. Select all columns that contain "Lifespan"
print(animals.select(cs.contains("Lifespan")))
```
</details>

## Expressions

Expressions are the heart of Polars. They are used to perform operations on columns.
You've already used them in the previous exercises!
`pl.col("Born") > 1900` is an expression.
`pl.col("Died") - pl.col("Born")` is an expression.

Let's try a more complex one.
From the `animals` dataset, create a new column `Lifespan_Diff_From_Mean` which is the `Average_Lifespan_years` minus the average lifespan of all animals in the dataset.

In [ ]:
# Your code here

### Namespaces

Polars expressions have specialized namespaces for specific data types.
- `.str` for string operations
- `.dt` for datetime operations
- `.list` for list operations
- `.struct` for struct operations

You can access these namespaces on a column expression. For example: `pl.col("name").str.to_uppercase()`.

Using the `animals` DataFrame:
1. Create a new column `Animal_Upper` which is the `Animal` name in uppercase.
2. Create a new column `Genus` which is the first word of the `Scientific_Name`.

In [ ]:
# Your code here

### Datetime Operations

Polars has first-class support for datetimes via the `.dt` namespace.

Using the `taxi` DataFrame (the parquet one you read earlier):
1. Create a new column `pickup_hour` that extracts the hour from `tpep_pickup_datetime`.
2. Create a new column `day_of_week` that extracts the day of the week (Monday=1, Sunday=7).

In [ ]:
# Your code here

### Logic: When / Then / Otherwise

To perform conditional logic, use `when().then().otherwise()`.
This is similar to if-else statements or SQL CASE WHEN.

Syntax:
```python
pl.when(condition) \
.then(value_if_true) \
.otherwise(value_if_false)
```

You can chain multiple `.when().then()` calls.

Using the `animals` DataFrame, create a `Lifespan_Category` column:
- "Short" if `Average_Lifespan_years` < 15
- "Medium" if `Average_Lifespan_years` is between 15 and 25 (inclusive)
- "Long" if `Average_Lifespan_years` > 25
- "Unknown" otherwise

In [ ]:
# Your code here

### Nulls and Sorting

Real world data is often messy and contains missing values (Nulls).
Polars provides methods to handle them:
- `fill_null(value)`: Replace nulls with a value.
- `drop_nulls()`: Remove rows with nulls.

Sorting is done with `.sort("column_name", descending=True/False)`.

Using the `animals` DataFrame:
1. Fill the null values in `Average_Weight_kg` with the mean weight of all animals.
2. Sort the result by `Average_Weight_kg` in descending order.

In [ ]:
# Your code here

## Window Functions

Window functions allow you to perform aggregations without grouping the entire DataFrame. This is done using the `.over()` expression.
It's like a "group_by, agg, and join back" all in one line.

Tasks using the `animals` DataFrame:
1. Create a column `Mean_Lifespan_By_Status` which is the average lifespan *per* `Conservation_Status`.
2. Create a column `Rank_Weight_By_Status` which ranks the animals by `Average_Weight_kg` within each `Conservation_Status`.

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Namespaces</summary>

```python
animals.with_columns(
    (pl.col("Average_Lifespan_years") - pl.col("Average_Lifespan_years").mean()).alias("Lifespan_Diff_From_Mean")
)
```
</details>

<details>
<summary>Show Answer: Datetime Operations</summary>

```python
animals.with_columns(
    pl.col("Animal").str.to_uppercase().alias("Animal_Upper"),
    pl.col("Scientific_Name").str.split(" ").list.get(0).alias("Genus"),
)
```
</details>

<details>
<summary>Show Answer: Logic: When / Then / Otherwise</summary>

```python
taxi_parquet.with_columns(
    pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
    pl.col("tpep_pickup_datetime").dt.weekday().alias("day_of_week"),
).select("tpep_pickup_datetime", "pickup_hour", "day_of_week").head()
```
</details>

<details>
<summary>Show Answer: Nulls and Sorting</summary>

```python
animals.with_columns(
    pl.when(pl.col("Average_Lifespan_years") < 15)
    .then(pl.lit("Short"))
    .when(pl.col("Average_Lifespan_years") <= 25)
    .then(pl.lit("Medium"))
    .when(pl.col("Average_Lifespan_years") > 25)
    .then(pl.lit("Long"))
    .otherwise(pl.lit("Unknown"))
    .alias("Lifespan_Category")
)
```
</details>

<details>
<summary>Show Answer: Dropping Nulls</summary>

```python
animals.with_columns(
    pl.col("Average_Weight_kg").fill_null(pl.mean("Average_Weight_kg"))
).sort("Average_Weight_kg", descending=True)
```
</details>

<details>
<summary>Show Answer: Window Functions</summary>

```python
animals.with_columns(
    pl.col("Average_Lifespan_years").mean().over("Conservation_Status").alias("Mean_Lifespan_By_Status"),
    pl.col("Average_Weight_kg").rank(descending=True).over("Conservation_Status").alias("Rank_Weight_By_Status"),
).select("Animal", "Conservation_Status", "Mean_Lifespan_By_Status", "Rank_Weight_By_Status")
```
</details>


## Case study: Yellow taxi dataset

Let's use the yellow taxi dataset to do some basic analysis.
We have two months of data: January and February 2025.
Let's compare some aggregated statistics for these two months.
Calculate the average tip percentage `(tip amount / total amount) * 100 for every VendorID, and find out which Vendor had the biggest improvement in tip percentage, going from january to february.

In [ ]:
# Your code here

### Answers

<details>
<summary>Show Answer: Morning Capstone</summary>

```python
# Create a function we can apply on both frames
def vendor_tip_percentage(path: str, month: str) -> pl.LazyFrame:
    return (
        pl.scan_parquet(path)
        .select(
            pl.col("VendorID"),
            pl.col("total_amount"),
            pl.col("tip_amount"),
        )
        .group_by("VendorID")
        .agg(
            (
                pl.col("tip_amount").sum() / pl.col("total_amount").sum().replace(0, None) * 100
            ).alias(f"tip_percentage_{month}")
        )
    )


# Process each month independently
jan = vendor_tip_percentage(
    "data/yellow_tripdata_2025-01.parquet",
    "january",
)

feb = vendor_tip_percentage(
    "data/yellow_tripdata_2025-02.parquet",
    "february",
)

# Join on VendorID, compute the increase and filter for the max
result = (
    jan.join(feb, on="VendorID", how="inner")
    .with_columns(
        (
            pl.col("tip_percentage_february")
            - pl.col("tip_percentage_january")
        ).alias("increase")
    )
    .filter(pl.col("increase") == pl.col("increase").max())
)

result.collect()
```
</details>
